# 04 — Error Analysis

Analyze where the model fails by examining misclassified samples.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("../outputs/results")
TEST_CSV = Path("../data/processed/test.csv")

metrics = json.load(open(RESULTS_DIR / "metrics.json"))
print("Metrics:", json.dumps(metrics, indent=2))

## Misclassified Samples

Reload test predictions (if saved) or load the confusion matrix.

In [ ]:
# Re-run predictions to get misclassified samples
import torch
from model import TenglishModel
from dataset import TenglishDataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from utils import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TenglishModel().to(device)
ckpt = load_checkpoint(model, None, "../outputs/checkpoints/best_model.pt")
model.eval()

test_ds = TenglishDataset(TEST_CSV)
test_loader = DataLoader(test_ds, batch_size=32)

y_true, y_pred = [], []
texts = []
@torch.no_grad()
def predict():
    for batch in test_loader:
        view1_ids = batch["view1_input_ids"].to(device)
        view1_mask = batch["view1_attention_mask"].to(device)
        view2_ids = batch["view2_input_ids"].to(device)
        view2_mask = batch["view2_attention_mask"].to(device)
        labels = batch["label"]
        
        logits = model(view1_ids, view1_mask, view2_ids, view2_mask)["logits"]
        preds = logits.argmax(1).cpu()
        y_true.extend(labels)
        y_pred.extend(preds)
predict()

errors = [i for i, (t, p) in enumerate(zip(y_true, y_pred)) if t != p]
print(f"Total errors: {len(errors)} / {len(y_true)}")

In [ ]:
# Show top error patterns
error_df = test_ds.df.iloc[errors].copy()
error_df["predicted"] = [y_pred[i] for i in errors]

LABEL_NAMES = ["positive", "negative", "neutral"]
error_df["true_label"] = error_df["label"].map(lambda x: LABEL_NAMES[x])
error_df["pred_label"] = error_df["predicted"].map(lambda x: LABEL_NAMES[x])

print("Most common error types:")
print(error_df.groupby(["true_label", "pred_label"]).size().sort_values(ascending=False).head(10))

print("\nSample misclassified sentences:")
print(error_df[["text_roman", "true_label", "pred_label"]].head(10).to_string())